<a href="https://colab.research.google.com/github/B-Mohid/AI_agents-automation_assignment/blob/main/TirthYatra.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# Install required libraries in Colab:
!pip install anthropic pydantic numpy

import os
import json
import numpy as np
from pydantic import BaseModel, Field
from anthropic import Anthropic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 14.7 MB/s eta 0:00:00


In [3]:
# 1. Input and Output Schemas
class PilgrimageParty(BaseModel):
    destination: str                       # e.g., "Tirupati", "Varanasi", "Kedarnath"
    party_size: int
    oldest_member_age: int
    has_wheelchair_or_palki: bool
    dietary_restriction: str              # "Jain", "Pure Veg Sattvik", "Vrat Fasting", "Standard Veg"
    darshan_slot_time: str                # e.g., "06:00 AM"
    hotel_checkin_time: str               # e.g., "12:00 PM"
    walking_distance_km: float
    stair_count: int

class LLMHospitalityEvaluation(BaseModel):
    dietary_compliance_score: float = Field(description="0.0 (fully compliant Sattvik kitchen) to 1.0 (high risk)")
    recommended_amenities: list[str]
    suggested_schedule_adjustments: list[str]
    hospitality_reasoning: str

In [5]:
# 2. Mathematical Strain Engine
def compute_yatra_fatigue_index(party: PilgrimageParty, llm_eval: LLMHospitalityEvaluation) -> dict:
    w1, w2, w3 = 0.50, 0.30, 0.20

    # Calculate Mobility Strain
    support_factor = 0.3 if party.has_wheelchair_or_palki else 1.0
    age_capacity = max(1.0, 90.0 - party.oldest_member_age)

    raw_mobility = ((party.walking_distance_km * 1.5) + (party.stair_count * 0.05)) * support_factor
    m_mobility = min(1.0, raw_mobility / age_capacity)

    # Calculate Dietary Penalty
    d_diet = llm_eval.dietary_compliance_score

    # Time Slack Penalty (Simplified heuristic for morning darshan vs noon checkin)
    t_slack_penalty = 0.15 if party.darshan_slot_time.startswith("0") else 0.40

    # Composite Score
    f_yatra = (w1 * m_mobility) + (w2 * d_diet) + (w3 * t_slack_penalty)
    f_yatra = round(float(f_yatra), 3)

    status = (
        "FEASIBLE (OPTIMAL COMFORT)" if f_yatra < 0.30
        else "MODERATE STRAIN (ADJUSTMENTS NEEDED)" if f_yatra < 0.60
        else "CRITICAL FATIGUE RISK (REPLAN REQUIRED)"
    )

    return {
        "fatigue_index": f_yatra,
        "status": status,
        "breakdown": {
            "mobility_strain_score": round(m_mobility, 2),
            "dietary_risk_score": round(d_diet, 2),
            "timing_friction_score": round(t_slack_penalty, 2)
        }
    }

In [6]:
# 3. Agent Execution Engine
class TirthYatraHostAgent:
    def __init__(self, api_key: str):
        self.client = Anthropic(api_key=api_key)

    def plan_spiritual_stay(self, party: PilgrimageParty) -> dict:
        prompt = f"""
        You are an expert Indian Pilgrimage Hospitality & Micro-Logistics Agent.
        Evaluate the lodging, dietary, and physical feasibility for this group:

        - Destination: {party.destination}
        - Party Size: {party.party_size} (Oldest Member: {party.oldest_member_age} years)
        - Mobility Support: {"Wheelchair/Palki booked" if party.has_wheelchair_or_palki else "None"}
        - Dietary Rules: {party.dietary_restriction}
        - Darshan Time: {party.darshan_slot_time} | Hotel Check-in: {party.hotel_checkin_time}
        - Physical Exertion: {party.walking_distance_km} km walking, {party.stair_count} steps

        Analyze local micro-hospitality constraints (e.g., Dharamshala vs Hotel proximity to Temple Gate, Sattvik food availability, senior accessibility).

        Return STRICTLY a JSON object matching this schema:
        {{
            "dietary_compliance_score": <float 0.0 (safe) to 1.0 (high risk)>,
            "recommended_amenities": [<list of strings, e.g. "Ground floor room near Gate 2", "In-house Sattvik kitchen">],
            "suggested_schedule_adjustments": [<list of actionable timing shifts>],
            "hospitality_reasoning": "<concise assessment>"
        }}
        """

        response = self.client.messages.create(
            model="claude-3-5-sonnet-20241022",
            max_tokens=1000,
            temperature=0.1,
            messages=[{"role": "user", "content": prompt}]
        )

        ai_analysis = json.loads(response.content[0].text)

        llm_eval = LLMHospitalityEvaluation(
            dietary_compliance_score=ai_analysis["dietary_compliance_score"],
            recommended_amenities=ai_analysis["recommended_amenities"],
            suggested_schedule_adjustments=ai_analysis["suggested_schedule_adjustments"],
            hospitality_reasoning=ai_analysis["hospitality_reasoning"]
        )

        math_results = compute_yatra_fatigue_index(party, llm_eval)

        return {
            "group_profile": party.model_dump(),
            "mathematical_fatigue_evaluation": math_results,
            "hospitality_plan": {
                "recommended_amenities": llm_eval.recommended_amenities,
                "schedule_adjustments": llm_eval.suggested_schedule_adjustments,
                "expert_assessment": llm_eval.hospitality_reasoning
            }
        }

In [12]:
# 4. Trial Execution
if __name__ == "__main__":
    API_KEY = os.getenv("ANTHROPIC_API_KEY", "your-claude-api-key")

    test_party = PilgrimageParty(
        destination="Varanasi (Kashi Vishwanath Corridor)",
        party_size=6,
        oldest_member_age=74,
        has_wheelchair_or_palki=False,
        dietary_restriction="Jain (No onion, no garlic, root vegetables restricted)",
        darshan_slot_time="05:00 AM",
        hotel_checkin_time="12:00 PM",
        walking_distance_km=3.2,
        stair_count=120
    )

    agent = TirthYatraHostAgent(api_key=API_KEY)
    #result = agent.plan_spiritual_stay(test_party)
    #print(json.dumps(result, indent=2))